# RF-DETR Fine-Tuning on Custom Dataset

This notebook fine-tunes an **RF-DETR** (Roboflow Detection Transformer) model on your own images labeled with **LabelMe**.

**This version combines 3 separate datasets** into one merged training set:
- `aruco_and_tray_1` (119 labeled images)
- `fixed_belt_view_1` (133 labeled images)
- `mixed_tray_belt_1` (98 labeled images)

**Combined:** 350 labeled images, 624 annotations across 5 classes (DM, HOUSING, EXCHANGER, CORE, SHAFT).

**Requirements before starting:**
- Your images labeled with LabelMe (one `.json` file per image)
- 3 zip files, one per dataset, each containing `.png`/`.jpg` images and their `.json` annotation files
- Google Colab with **T4 GPU** runtime selected

**Pipeline:**
1. Upload & extract 3 dataset zips, merge into one labeled dataset (with prefixed filenames)
2. Convert LabelMe annotations to COCO JSON format
3. Split into train/validation sets & organize folder structure
4. Train RF-DETR (Base model, T4-optimized settings)
5. Evaluate & visualize results
6. Download trained model

## 0. GPU Check & Setup

In [ ]:
# Verify GPU is available
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected! Go to Runtime > Change runtime type > T4 GPU"
    )

In [ ]:
# Install dependencies
!pip install -q rfdetr labelme2coco supervision scikit-learn

## 1. Load & Merge Labeled Data (3 Datasets)

Place your 3 zip files in `/content/labelme_data/` on Colab before running this section.

Expected zip files:
```
/content/labelme_data/
  aruco_and_tray_1.zip       (119 labeled + 15 unlabeled images)
  fixed_belt_view_1.zip      (133 labeled + 51 unlabeled images)
  mixed_tray_belt_1.zip      (98 labeled images)
```

Filenames are prefixed during merge to avoid collisions (e.g., `at1_image_000.png`).
Only images that have **both** a `.png` and `.json` file are included (unlabeled images are skipped).

### Extract & Merge Datasets

In [ ]:
# Extract 3 zip files from /content/labelme_data/ and merge into one dataset
import zipfile
import os
import json
import shutil

# Zips are already uploaded to this directory
ZIP_DIR = "/content/labelme_data"
# Merged output goes to a separate directory (avoids collision with zip source)
LABELME_DIR = "/content/labelme_merged"
os.makedirs(LABELME_DIR, exist_ok=True)

# Define datasets and their filename prefixes
DATASETS = {
    "aruco_and_tray_1": "at1",
    "fixed_belt_view_1": "fbv1",
    "mixed_tray_belt_1": "mtb1",
}

# Find zip files in ZIP_DIR
available_zips = [f for f in os.listdir(ZIP_DIR) if f.endswith(".zip")]
print(f"Found {len(available_zips)} zip files in {ZIP_DIR}:")
for z in available_zips:
    size_mb = os.path.getsize(os.path.join(ZIP_DIR, z)) / 1024 / 1024
    print(f"  {z} ({size_mb:.1f} MB)")

# Auto-match zip files to dataset names
dataset_zips = {}
for zip_name in available_zips:
    for ds_name in DATASETS:
        if ds_name in zip_name.lower().replace("-", "_").replace(" ", "_"):
            dataset_zips[ds_name] = zip_name
            break

if len(dataset_zips) != 3:
    matched = list(dataset_zips.keys())
    missing = [ds for ds in DATASETS if ds not in matched]
    print(f"\nERROR: Could not auto-match all 3 datasets.")
    print(f"Matched: {matched}")
    print(f"Missing: {missing}")
    print(f"\nManually set dataset_zips, e.g.:")
    print("dataset_zips = {")
    for ds in DATASETS:
        print(f'    "{ds}": "your_zip_name.zip",')
    print("}")
    raise ValueError("Fix the dataset_zips mapping above and re-run this cell")

print(f"\nDataset mapping:")
for ds, zf in dataset_zips.items():
    print(f"  {ds} -> {zf}")

# Extract and merge with prefixed filenames
total_images = 0
total_jsons = 0
skipped = 0

for ds_name, prefix in DATASETS.items():
    zip_name = dataset_zips[ds_name]
    zip_path = os.path.join(ZIP_DIR, zip_name)

    # Extract to temp dir
    temp_dir = f"/content/temp_{prefix}"
    os.makedirs(temp_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(temp_dir)

    # Handle single subfolder in zip
    entries = os.listdir(temp_dir)
    src_dir = temp_dir
    if len(entries) == 1 and os.path.isdir(os.path.join(temp_dir, entries[0])):
        src_dir = os.path.join(temp_dir, entries[0])

    # Find labeled images (those with both .png and .json)
    all_pngs = [f for f in os.listdir(src_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    all_jsons = set(f.replace(".json", "") for f in os.listdir(src_dir) if f.endswith(".json"))

    ds_images = 0
    ds_jsons = 0
    ds_skipped = 0

    for png in sorted(all_pngs):
        stem = os.path.splitext(png)[0]
        ext = os.path.splitext(png)[1]

        if stem not in all_jsons:
            ds_skipped += 1
            skipped += 1
            continue

        # New prefixed names
        new_png = f"{prefix}_{stem}{ext}"
        new_json = f"{prefix}_{stem}.json"

        # Copy image
        shutil.copy2(os.path.join(src_dir, png), os.path.join(LABELME_DIR, new_png))
        ds_images += 1
        total_images += 1

        # Copy and update JSON (fix imagePath)
        json_path = os.path.join(src_dir, f"{stem}.json")
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        data["imagePath"] = new_png
        with open(os.path.join(LABELME_DIR, new_json), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        ds_jsons += 1
        total_jsons += 1

    # Clean up temp dir
    shutil.rmtree(temp_dir)

    print(f"  {ds_name} ({prefix}): {ds_images} images, {ds_jsons} JSONs, {ds_skipped} skipped (unlabeled)")

print(f"\nCombined dataset: {total_images} images, {total_jsons} JSONs")
print(f"Skipped {skipped} unlabeled images total")
print(f"Output: {LABELME_DIR}")

## 2. Convert LabelMe to COCO JSON Format

RF-DETR expects COCO-format annotations. We convert the per-image LabelMe JSONs into a single COCO annotation file.

In [ ]:
import labelme2coco

COCO_OUTPUT_DIR = "/content/coco_output"

# Convert all LabelMe annotations into a single COCO JSON
# category_id_start=1 is important -- RF-DETR expects 1-indexed category IDs
labelme2coco.convert(
    labelme_folder=LABELME_DIR,
    export_dir=COCO_OUTPUT_DIR,
    train_split_rate=1.0,       # don't split yet, we do it manually for more control
    category_id_start=1,
)

print(f"\nCOCO annotation file created at: {COCO_OUTPUT_DIR}/dataset.json")

In [ ]:
# Inspect the COCO annotations
import json

with open(f"{COCO_OUTPUT_DIR}/dataset.json") as f:
    coco_data = json.load(f)

print(f"Images:      {len(coco_data['images'])}")
print(f"Annotations: {len(coco_data['annotations'])}")
print(f"Categories:  {len(coco_data['categories'])}")
print(f"\nCategories:")
for cat in coco_data["categories"]:
    count = sum(1 for a in coco_data["annotations"] if a["category_id"] == cat["id"])
    print(f"  ID={cat['id']}: '{cat['name']}' -- {count} annotations")

avg_anns = len(coco_data["annotations"]) / max(len(coco_data["images"]), 1)
print(f"\nAvg annotations per image: {avg_anns:.1f}")

## 3. Train/Validation Split & Folder Structure

We split the dataset 85% train / 15% validation, then organize into the folder structure RF-DETR expects.

In [ ]:
import json
import os
import shutil
from sklearn.model_selection import train_test_split

# -- Configuration --
VAL_RATIO = 0.15       # 15% for validation
RANDOM_SEED = 42       # reproducible split
DATASET_DIR = "/content/dataset"  # final dataset directory for RF-DETR

# -- Load the COCO JSON --
with open(f"{COCO_OUTPUT_DIR}/dataset.json") as f:
    coco = json.load(f)

images = coco["images"]
annotations = coco["annotations"]
categories = coco["categories"]

# -- Split at image level --
train_imgs, val_imgs = train_test_split(
    images, test_size=VAL_RATIO, random_state=RANDOM_SEED
)

train_img_ids = {img["id"] for img in train_imgs}
val_img_ids = {img["id"] for img in val_imgs}

train_anns = [a for a in annotations if a["image_id"] in train_img_ids]
val_anns = [a for a in annotations if a["image_id"] in val_img_ids]

print(f"Train: {len(train_imgs)} images, {len(train_anns)} annotations")
print(f"Val:   {len(val_imgs)} images, {len(val_anns)} annotations")

# -- Create folder structure --
for split in ["train", "valid"]:
    os.makedirs(f"{DATASET_DIR}/{split}", exist_ok=True)

# -- Helper: fix file_name paths (labelme2coco may use absolute paths) --
def get_basename(file_name):
    """Extract just the filename from potentially absolute paths."""
    return os.path.basename(file_name)

# -- Copy images and create annotation files for each split --
for split_name, split_imgs, split_anns in [
    ("train", train_imgs, train_anns),
    ("valid", val_imgs, val_anns),
]:
    split_dir = f"{DATASET_DIR}/{split_name}"

    # Fix file_name to be just the basename
    for img in split_imgs:
        img["file_name"] = get_basename(img["file_name"])

    # Copy image files
    copied = 0
    for img in split_imgs:
        src = os.path.join(LABELME_DIR, img["file_name"])
        dst = os.path.join(split_dir, img["file_name"])
        if os.path.exists(src):
            shutil.copy2(src, dst)
            copied += 1
        else:
            print(f"  WARNING: Image not found: {src}")

    # Write COCO annotation JSON
    split_coco = {
        "images": split_imgs,
        "annotations": split_anns,
        "categories": categories,
    }
    ann_path = os.path.join(split_dir, "_annotations.coco.json")
    with open(ann_path, "w") as f:
        json.dump(split_coco, f, indent=2)

    print(f"  {split_name}: copied {copied} images, saved {ann_path}")

# -- Verify --
print(f"\nDataset ready at: {DATASET_DIR}")
for split in ["train", "valid"]:
    n_files = len(os.listdir(f"{DATASET_DIR}/{split}"))
    print(f"  {split}/: {n_files} files (images + annotations.json)")

## 4. Train RF-DETR

We use **RFDETRBase** (29M params) -- the best balance of accuracy and VRAM usage for a T4 GPU.

**Dataset:** 350 images across 5 classes (DM, HOUSING, EXCHANGER, CORE, SHAFT) with 624 annotations.

**Settings optimized for your setup:**
- `batch_size=4` with `grad_accum_steps=4` -> effective batch size of 16
- `epochs=60` -- good for 350 images with strong DINOv2 backbone transfer
- Early stopping enabled to prevent overfitting
- EMA (Exponential Moving Average) enabled for better generalization

> **Note:** Transformer models may show little progress in the first 5-10 epochs, then suddenly improve. This is normal!

In [ ]:
# -- Training configuration --
# Adjust these if needed:

EPOCHS = 60              # 350 images with 5 classes — good dataset size
BATCH_SIZE = 4           # fits T4 16GB VRAM comfortably
GRAD_ACCUM_STEPS = 4     # effective batch size = 4 * 4 = 16
LEARNING_RATE = 1e-4     # default, works well for fine-tuning
OUTPUT_DIR = "/content/training_output"

In [ ]:
from rfdetr import RFDETRBase

model = RFDETRBase()

model.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    lr=LEARNING_RATE,
    output_dir=OUTPUT_DIR,

    # Regularization & augmentation
    use_ema=True,
    multi_scale=True,

    # Early stopping -- prevents overfitting on small datasets
    early_stopping=True,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,

    # Logging
    tensorboard=True,
    checkpoint_interval=10,
    log_per_class_metrics=True,

    # Workers -- Colab has limited CPU cores
    num_workers=2,
)

print("\n" + "=" * 60)
print("Training complete!")
print(f"Best model saved at: {OUTPUT_DIR}/checkpoint_best_total.pth")
print("=" * 60)

## 5. Evaluate & Visualize Results

In [ ]:
# Load the best trained model
from rfdetr import RFDETRBase

trained_model = RFDETRBase(
    pretrain_weights=f"{OUTPUT_DIR}/checkpoint_best_total.pth"
)

In [ ]:
# Check training log for metrics
import os
import json

log_path = f"{OUTPUT_DIR}/log.txt"
if os.path.exists(log_path):
    print("Training log (last 5 entries):")
    with open(log_path) as f:
        lines = f.readlines()
    for line in lines[-5:]:
        try:
            entry = json.loads(line.strip())
            epoch = entry.get("epoch", "?")
            coco_metrics = entry.get("test_coco_eval_bbox", [])
            ema_metrics = entry.get("ema_test_coco_eval_bbox", [])
            map50_95 = f"{coco_metrics[0]:.4f}" if len(coco_metrics) > 0 else "N/A"
            map50 = f"{coco_metrics[1]:.4f}" if len(coco_metrics) > 1 else "N/A"
            ema_map = f"{ema_metrics[0]:.4f}" if len(ema_metrics) > 0 else "N/A"
            print(f"  Epoch {epoch}: mAP@50:95={map50_95}  mAP@50={map50}  EMA-mAP={ema_map}")
        except Exception:
            pass
else:
    print("No log.txt found")

results_path = f"{OUTPUT_DIR}/results.json"
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print("\nFinal evaluation results:")
    print(json.dumps(results, indent=2))

In [ ]:
# Visualize predictions on validation images
import supervision as sv
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import random

# Load class names from the dataset
with open(f"{DATASET_DIR}/valid/_annotations.coco.json") as f:
    val_coco = json.load(f)

class_names = {cat["id"]: cat["name"] for cat in val_coco["categories"]}

# Get validation images
val_dir = f"{DATASET_DIR}/valid"
val_images = [
    f for f in os.listdir(val_dir)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

# Show predictions on up to 8 random validation images
n_show = min(8, len(val_images))
sample_images = random.sample(val_images, n_show)

cols = min(4, n_show)
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
if n_show == 1:
    axes = [axes]
else:
    axes = axes.flatten()

box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)

for idx, img_name in enumerate(sample_images):
    img_path = os.path.join(val_dir, img_name)
    image = Image.open(img_path)

    detections = trained_model.predict(image, threshold=0.5)

    labels = []
    for class_id, conf in zip(detections.class_id, detections.confidence):
        name = class_names.get(class_id, f"class_{class_id}")
        labels.append(f"{name} {conf:.2f}")

    img_array = np.array(image)
    annotated = box_annotator.annotate(img_array.copy(), detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)

    axes[idx].imshow(annotated)
    axes[idx].set_title(f"{img_name}\n{len(detections)} detections", fontsize=9)
    axes[idx].axis("off")

for idx in range(n_show, len(axes)):
    axes[idx].axis("off")

plt.suptitle("Validation Set Predictions (threshold=0.5)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Download Trained Model

Download the best checkpoint so you can use it locally or deploy it.

In [ ]:
# Check checkpoint file sizes
import os

print("Saved checkpoints:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
# Download the best model checkpoint
from google.colab import files

best_checkpoint = f"{OUTPUT_DIR}/checkpoint_best_total.pth"
if os.path.exists(best_checkpoint):
    print(f"Downloading: {best_checkpoint}")
    files.download(best_checkpoint)
else:
    print("Best checkpoint not found. Available files:")
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith(".pth"):
            print(f"  {f}")

### Optional: Save to Google Drive

In [ ]:
# Optional: Copy best model to Google Drive for safekeeping
# Uncomment the lines below to use:

# from google.colab import drive
# drive.mount("/content/drive")
#
# import shutil
# drive_dest = "/content/drive/MyDrive/rf-detr-trained/"
# os.makedirs(drive_dest, exist_ok=True)
# shutil.copy2(best_checkpoint, drive_dest)
# print(f"Model saved to Google Drive: {drive_dest}")

## 7. Run Inference on New Images (Optional)

Upload a new image and run the trained model on it.

In [ ]:
# Upload and run inference on a new image
from google.colab import files
from PIL import Image
import supervision as sv
import numpy as np
import matplotlib.pyplot as plt

print("Upload an image to run inference on...")
uploaded = files.upload()

for filename in uploaded.keys():
    image = Image.open(filename)

    # Run inference (lower threshold to see more detections)
    detections = trained_model.predict(image, threshold=0.3)

    # Annotate
    labels = []
    for class_id, conf in zip(detections.class_id, detections.confidence):
        name = class_names.get(class_id, f"class_{class_id}")
        labels.append(f"{name} {conf:.2f}")

    img_array = np.array(image)
    box_annotator = sv.BoxAnnotator(thickness=2)
    label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)
    annotated = box_annotator.annotate(img_array.copy(), detections)
    annotated = label_annotator.annotate(annotated, detections, labels=labels)

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated)
    plt.title(f"{filename} -- {len(detections)} detections")
    plt.axis("off")
    plt.show()

    print(f"\n{filename}: {len(detections)} detections")
    for i, (class_id, conf, bbox) in enumerate(
        zip(detections.class_id, detections.confidence, detections.xyxy)
    ):
        name = class_names.get(class_id, f"class_{class_id}")
        print(f"  [{i+1}] {name}: {conf:.3f} at [{bbox[0]:.0f}, {bbox[1]:.0f}, {bbox[2]:.0f}, {bbox[3]:.0f}]")

## 8. Export to ONNX (Optional)

Export the trained model to ONNX format for deployment.

In [ ]:
# Export to ONNX
ONNX_OUTPUT_DIR = "/content/onnx_export"

trained_model.export(
    output_dir=ONNX_OUTPUT_DIR,
    opset_version=17,
)

print(f"\nONNX model exported to: {ONNX_OUTPUT_DIR}")
for f in os.listdir(ONNX_OUTPUT_DIR):
    size_mb = os.path.getsize(os.path.join(ONNX_OUTPUT_DIR, f)) / 1024 / 1024
    print(f"  {f}: {size_mb:.1f} MB")

# Download ONNX model
from google.colab import files
onnx_files = [f for f in os.listdir(ONNX_OUTPUT_DIR) if f.endswith(".onnx")]
if onnx_files:
    files.download(os.path.join(ONNX_OUTPUT_DIR, onnx_files[0]))